# SGG YOLO CONVERTER

### This script can convert any SGG annotations file in COCO format (PSG dataset) to YOLO format for object detection training

In [11]:
import os
import shutil
import random
import h5py
import json
from collections import Counter
from tqdm import tqdm

# Load PSG annotations
data = json.load(open('../datasets/psg/psg/psg_train_val.json'))

# Path to COCO images
coco_path = "../datasets/psg/coco/coco/"

In [ ]:
# Skip this cell - images are already in correct location

100%|██████████| 48749/48749 [00:51<00:00, 942.29it/s] 


In [12]:
# Test reading an image
import cv2
test_img_path = os.path.join(coco_path, data['data'][0]['file_name'])
img = cv2.imread(test_img_path)
print(f"Image shape: {img.shape}")
print(f"Image path: {test_img_path}")

Image shape: (640, 480, 3)
Image path: ../datasets/psg/coco/coco/train2017/000000417720.jpg


In [13]:
object_to_idx = {}
THING_CLASSES = data['thing_classes']
STUFF_CLASSES = data['stuff_classes']
CLASSES = THING_CLASSES + STUFF_CLASSES
for i, cls in enumerate(CLASSES):
    object_to_idx[cls] = str(i)

OUT_PATH = "../datasets/psg/YOLO_anno/"
if not os.path.exists(OUT_PATH):
    os.makedirs(OUT_PATH)

with open(os.path.join(OUT_PATH, 'classes.txt'), 'w') as f:
    for key,v in object_to_idx.items():
        f.write(key + '\n')

print(f"Total classes: {len(CLASSES)}")
print(f"Thing classes: {len(THING_CLASSES)}, Stuff classes: {len(STUFF_CLASSES)}")

Total classes: 133
Thing classes: 80, Stuff classes: 53


In [14]:
print(THING_CLASSES)

['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed', 'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']


In [15]:
print(STUFF_CLASSES)

['banner', 'blanket', 'bridge', 'cardboard', 'counter', 'curtain', 'door-stuff', 'floor-wood', 'flower', 'fruit', 'gravel', 'house', 'light', 'mirror-stuff', 'net', 'pillow', 'platform', 'playingfield', 'railroad', 'river', 'road', 'roof', 'sand', 'sea', 'shelf', 'snow', 'stairs', 'tent', 'towel', 'wall-brick', 'wall-stone', 'wall-tile', 'wall-wood', 'water-other', 'window-blind', 'window-other', 'tree-merged', 'fence-merged', 'ceiling-merged', 'sky-other-merged', 'cabinet-merged', 'table-merged', 'floor-other-merged', 'pavement-merged', 'mountain-merged', 'grass-merged', 'dirt-merged', 'paper-merged', 'food-other-merged', 'building-other-merged', 'rock-merged', 'wall-other-merged', 'rug-merged']


In [16]:
dest_folder = os.path.join(OUT_PATH, 'annotations')
image_dest_folder = os.path.join(OUT_PATH, 'images_all')

import numpy as np

if not os.path.exists(dest_folder):
    os.makedirs(dest_folder)
if not os.path.exists(image_dest_folder):
    os.makedirs(image_dest_folder)

for idx in tqdm(range(len(data['data']))):
    image_data = data['data']
    img_width = image_data[idx]['width']
    img_height = image_data[idx]['height']
    image_id = image_data[idx]['image_id']
    file_name = image_data[idx]['file_name']

    anno = image_data[idx]['annotations']

    for obj in anno:
        obj_cat = obj['category_id']
        assert 0 <= obj_cat < len(CLASSES), f"Category {obj_cat} out of range"

        box = obj['bbox']
        # convert to yolo format
        x_center = (box[0] + box[2] / 2) / img_width
        y_center = (box[1] + box[3] / 2) / img_height
        w = box[2] / img_width
        h = box[3] / img_height

        box = [x_center, y_center, w, h]

        # clip to [0, 1]
        box = np.clip(box, 0, 1)

        with open(os.path.join(dest_folder, str(image_id) + '.txt'), 'a') as f:
            f.write(str(obj_cat) + ' ' + str(box[0]) + ' ' + str(box[1]) + ' ' + str(box[2]) + ' ' + str(box[3])+'\n')
    
    # copy image to dest folder
    src_img = os.path.join(coco_path, file_name)
    dst_img = os.path.join(image_dest_folder, str(image_id)+'.jpg')
    if os.path.exists(src_img):
        shutil.copyfile(src_img, dst_img)
    else:
        print(f"Warning: Image not found - {src_img}")

print(f"Conversion complete! Images and annotations saved to {OUT_PATH}")

100%|██████████| 46697/46697 [49:24<00:00, 15.75it/s]  



Conversion complete! Images and annotations saved to ../datasets/psg/YOLO_anno/


### With val split

In [17]:
from sklearn.model_selection import train_test_split

base_path = OUT_PATH
dest_folder = os.path.join(base_path,'annotations')
image_dest_folder = os.path.join(base_path,'images_all')

images = [os.path.join(image_dest_folder, x) for x in os.listdir(image_dest_folder) if x.endswith('.jpg')]
annotations = [os.path.join(dest_folder, x) for x in os.listdir(dest_folder) if x.endswith('.txt')]

images.sort()
annotations.sort()

print(f"Total images: {len(images)}, Total annotations: {len(annotations)}")

train_images, val_images, train_annotations, val_annotations = train_test_split(images, annotations, test_size = 0.3, random_state = 1)
val_images, test_images, val_annotations, test_annotations = train_test_split(val_images, val_annotations, test_size = 0.5, random_state = 1)

print(f"Train: {len(train_images)}, Val: {len(val_images)}, Test: {len(test_images)}")

root_path = 'images/'
folders = [base_path+'/train',base_path+'/test',base_path+'/val']
for folder in folders:
    os.makedirs(os.path.join(folder,root_path), exist_ok=True)
    
root_path = 'labels/'
for folder in folders:
    os.makedirs(os.path.join(folder,root_path), exist_ok=True)

def move_files_to_folder(list_of_files, destination_folder):
    for f in tqdm(list_of_files, desc=f"Moving to {destination_folder}"):
        try:
            shutil.move(f, destination_folder)
        except Exception as e:
            print(f"Error moving {f}: {e}")

move_files_to_folder(train_images, base_path+'/train/images/')
move_files_to_folder(val_images, base_path+'/val/images/')
move_files_to_folder(test_images, base_path+'/test/images/')
move_files_to_folder(train_annotations, base_path+'/train/labels/')
move_files_to_folder(val_annotations, base_path+'/val/labels/')
move_files_to_folder(test_annotations, base_path+'/test/labels/')

print("Split complete!")

Total images: 46697, Total annotations: 46697
Train: 32687, Val: 7005, Test: 7005


Moving to ../datasets/psg/YOLO_anno//train/images/: 100%|██████████| 32687/32687 [05:24<00:00, 100.76it/s]
Moving to ../datasets/psg/YOLO_anno//train/images/: 100%|██████████| 32687/32687 [05:24<00:00, 100.76it/s]
Moving to ../datasets/psg/YOLO_anno//val/images/: 100%|██████████| 7005/7005 [01:05<00:00, 106.47it/s]
Moving to ../datasets/psg/YOLO_anno//test/images/: 100%|██████████| 7005/7005 [01:08<00:00, 101.55it/s]
Moving to ../datasets/psg/YOLO_anno//test/images/: 100%|██████████| 7005/7005 [01:08<00:00, 101.55it/s]
Moving to ../datasets/psg/YOLO_anno//train/labels/: 100%|██████████| 32687/32687 [04:47<00:00, 113.83it/s]
Moving to ../datasets/psg/YOLO_anno//train/labels/: 100%|██████████| 32687/32687 [04:47<00:00, 113.83it/s]
Moving to ../datasets/psg/YOLO_anno//val/labels/: 100%|██████████| 7005/7005 [01:01<00:00, 114.59it/s]
Moving to ../datasets/psg/YOLO_anno//test/labels/: 100%|██████████| 7005/7005 [01:00<00:00, 115.41it/s]

Split complete!


### No val split

In [ ]:
from sklearn.model_selection import train_test_split

base_path = OUT_PATH
dest_folder = os.path.join(base_path,'annotations')
image_dest_folder = os.path.join(base_path,'images_all')

images = [os.path.join(image_dest_folder, x) for x in os.listdir(image_dest_folder) if x.endswith('.jpg')]
annotations = [os.path.join(dest_folder, x) for x in os.listdir(dest_folder) if x.endswith('.txt')]

images.sort()
annotations.sort()

print(f"Total images: {len(images)}, Total annotations: {len(annotations)}")

train_images, test_images, train_annotations, test_annotations = train_test_split(images, annotations, test_size = 0.2, random_state = 1)

print(f"Train: {len(train_images)}, Val: {len(test_images)}")

root_path = 'images/'
folders = [base_path+'/train',base_path+'/val']
for folder in folders:
    os.makedirs(os.path.join(folder,root_path), exist_ok=True)
    
root_path = 'labels/'
for folder in folders:
    os.makedirs(os.path.join(folder,root_path), exist_ok=True)

def move_files_to_folder(list_of_files, destination_folder):
    for f in tqdm(list_of_files, desc=f"Moving to {destination_folder}"):
        try:
            shutil.move(f, destination_folder)
        except Exception as e:
            print(f"Error moving {f}: {e}")

move_files_to_folder(train_images, base_path+'/train/images/')
move_files_to_folder(test_images, base_path+'/val/images/')
move_files_to_folder(train_annotations, base_path+'/train/labels/')
move_files_to_folder(test_annotations, base_path+'/val/labels/')

print("Split complete!")

In [18]:
# write the yaml file
classes = list(object_to_idx.keys())

train_path = os.path.abspath(base_path+'train/images')
val_path = os.path.abspath(base_path+'val/images')
test_path = os.path.abspath(base_path+'test/images') if os.path.exists(base_path+'test') else val_path

n_classes = len(classes)

yaml_file = base_path+'/data.yaml'
with open(yaml_file, 'w') as f:
    f.write(f'train: {train_path}\n')
    f.write(f'val: {val_path}\n')
    f.write(f'test: {test_path}\n')
    f.write(f'nc: {n_classes}\n')
    f.write(f'names: {classes}\n')

print(f"YAML file created: {yaml_file}")
print(f"Number of classes: {n_classes}")
print(f"Train path: {train_path}")
print(f"Val path: {val_path}")

YAML file created: ../datasets/psg/YOLO_anno//data.yaml
Number of classes: 133
Train path: /mnt/h/gdrive/Takeout/Drive/School/4 Fourth year/BCTN/code/sgg-feedback/sgg/datasets/psg/YOLO_anno/train/images
Val path: /mnt/h/gdrive/Takeout/Drive/School/4 Fourth year/BCTN/code/sgg-feedback/sgg/datasets/psg/YOLO_anno/val/images


### Train YOLO11 with converted dataset

In [ ]:
from ultralytics import YOLO
import json
import pandas as pd
from datetime import datetime

# Load YOLO11 model
model = YOLO('../../checkpoints/yolo11m.pt')  # or yolo11s/l/x

# Train
results = model.train(
    data=yaml_file,
    epochs=100,
    imgsz=640,
    batch=16,
    project='checkpoints/PSG',
    name='yolo11m_psg',
    device=0,
    patience=20,
    save=True,
    plots=True,
    # Ultralytics tự động lưu metrics vào runs/
)

print("Training complete!")
print(f"Best model saved at: checkpoints/PSG/yolo11m_psg/weights/best.pt")

# ============================================================
# EXTRACT & SAVE TRAINING METRICS
# ============================================================

# 1. Load metrics từ CSV file (tự động được tạo bởi Ultralytics)
results_csv = "checkpoints/PSG/yolo11m_psg/results.csv"
metrics_df = pd.read_csv(results_csv)

print(f"\n📊 Training Metrics saved to: {results_csv}")
print(f"   Columns: {list(metrics_df.columns)}")

# 2. Extract key metrics
training_summary = {
    "model": "yolo11m",
    "dataset": "PSG",
    "timestamp": datetime.now().isoformat(),
    "config": {
        "epochs": 100,
        "batch_size": 16,
        "img_size": 640,
        "patience": 20,
    },
    "final_metrics": {
        "mAP50": float(metrics_df['metrics/mAP50(B)'].iloc[-1]),
        "mAP50-95": float(metrics_df['metrics/mAP50-95(B)'].iloc[-1]),
        "precision": float(metrics_df['metrics/precision(B)'].iloc[-1]),
        "recall": float(metrics_df['metrics/recall(B)'].iloc[-1]),
        "train_loss": float(metrics_df['train/box_loss'].iloc[-1]),
        "val_loss": float(metrics_df['val/box_loss'].iloc[-1]),
    },
    "best_epoch": int(metrics_df['metrics/mAP50-95(B)'].idxmax() + 1),
    "total_epochs_trained": len(metrics_df),
}

# 3. Save summary to JSON
summary_file = "checkpoints/PSG/yolo11m_psg/training_summary.json"
with open(summary_file, 'w') as f:
    json.dump(training_summary, f, indent=2)

print(f"\n✅ Training Summary saved to: {summary_file}")
print(f"\n📈 Final Results:")
for key, value in training_summary['final_metrics'].items():
    print(f"   {key}: {value:.4f}")

# 4. Quick visualization preview
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# mAP
axes[0, 0].plot(metrics_df['epoch'], metrics_df['metrics/mAP50(B)'], label='mAP50', linewidth=2)
axes[0, 0].plot(metrics_df['epoch'], metrics_df['metrics/mAP50-95(B)'], label='mAP50-95', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('mAP')
axes[0, 0].set_title('Mean Average Precision')
axes[0, 0].legend()
axes[0, 0].grid(True)

# Losses
axes[0, 1].plot(metrics_df['epoch'], metrics_df['train/box_loss'], label='Train Box Loss', linewidth=2)
axes[0, 1].plot(metrics_df['epoch'], metrics_df['val/box_loss'], label='Val Box Loss', linewidth=2)
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].set_title('Box Loss')
axes[0, 1].legend()
axes[0, 1].grid(True)

# Precision & Recall
axes[1, 0].plot(metrics_df['epoch'], metrics_df['metrics/precision(B)'], label='Precision', linewidth=2)
axes[1, 0].plot(metrics_df['epoch'], metrics_df['metrics/recall(B)'], label='Recall', linewidth=2)
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Score')
axes[1, 0].set_title('Precision & Recall')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Class Loss
axes[1, 1].plot(metrics_df['epoch'], metrics_df['train/cls_loss'], label='Train Cls Loss', linewidth=2)
axes[1, 1].plot(metrics_df['epoch'], metrics_df['val/cls_loss'], label='Val Cls Loss', linewidth=2)
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].set_title('Classification Loss')
axes[1, 1].legend()
axes[1, 1].grid(True)

plt.tight_layout()
plt.savefig('checkpoints/PSG/yolo11m_psg/training_curves.png', dpi=150)
plt.show()

print(f"\n📊 Training curves saved to: checkpoints/PSG/yolo11m_psg/training_curves.png")

⚠️ Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11m.pt...
⚠️ Download failure, retrying 1/3 https://github.com/ultralytics/assets/releases/download/v8.3.0/yolo11m.pt...


######################################################################## 100.0%#                               60.4%#                               60.4%
######################################################################## 100.0%


New https://pypi.org/project/ultralytics/8.3.235 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.100 🚀 Python-3.11.14 torch-2.5.1+cu124 CUDA:0 (NVIDIA CMP 40HX, 8192MiB)
Ultralytics 8.3.100 🚀 Python-3.11.14 torch-2.5.1+cu124 CUDA:0 (NVIDIA CMP 40HX, 8192MiB)
engine/trainer: task=detect, mode=train, model=../../checkpoints/yolo11m.pt, data=../datasets/psg/YOLO_anno//data.yaml, epochs=100, time=None, patience=20, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=0, workers=8, project=checkpoints/PSG, name=yolo11m_psg, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stre

Fontconfig error: Cannot load default config file: No such file: (null)


Overriding model.yaml nc=80 with nc=133

                   from  n    params  module                                       arguments                     

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  2                  -1  1    111872  ultralytics.nn.modules.block.C3k2            [128, 256, 1, True, 0.25]     
  3                  -1  1    590336  ultralytics.nn.modules.conv.Conv             [256, 256, 3, 2]              
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  2                  -1  1    111872  ultralyt

...

train: WARNING ⚠️ /mnt/h/gdrive/Takeout/Drive/School/4 Fourth year/BCTN/code/sgg-feedback/sgg/datasets/psg/YOLO_anno/train/images/000000582095_jpg.rf.349e72ee32ee51f19265e477b1f46709.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /mnt/h/gdrive/Takeout/Drive/School/4 Fourth year/BCTN/code/sgg-feedback/sgg/datasets/psg/YOLO_anno/train/images/150308.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /mnt/h/gdrive/Takeout/Drive/School/4 Fourth year/BCTN/code/sgg-feedback/sgg/datasets/psg/YOLO_anno/train/images/2334926.jpg: 1 duplicate labels removed
train: WARNING ⚠️ /mnt/h/gdrive/Takeout/Drive/School/4 Fourth year/BCTN/code/sgg-feedback/sgg/datasets/psg/YOLO_anno/train/images/2408943.jpg: 1 duplicate labels removed


train: New cache created: /mnt/h/gdrive/Takeout/Drive/School/4 Fourth year/BCTN/code/sgg-feedback/sgg/datasets/psg/YOLO_anno/train/labels.cache


val: Scanning /mnt/h/gdrive/Takeout/Drive/School/4 Fourth year/BCTN/code/sgg-feedback/sgg/datasets/psg/YOLO_anno/val/labels... 7005 images, 0 backgrounds, 0 corrupt: 100%|██████████| 7005/7005 [00:09<00:00, 724.29it/s]

val: WARNING ⚠️ /mnt/h/gdrive/Takeout/Drive/School/4 Fourth year/BCTN/code/sgg-feedback/sgg/datasets/psg/YOLO_anno/val/images/2397126.jpg: 1 duplicate labels removed


val: New cache created: /mnt/h/gdrive/Takeout/Drive/School/4 Fourth year/BCTN/code/sgg-feedback/sgg/datasets/psg/YOLO_anno/val/labels.cache


### Detailed Analysis & Visualization

Load và phân tích chi tiết các metrics đã lưu

### Export to ONNX

In [ ]:
# Export trained model to ONNX
model = YOLO('checkpoints/PSG/yolo11m_psg/weights/best.pt')
model.export(format='onnx', opset=12, simplify=True, imgsz=640)

print("ONNX export complete!")
print("ONNX file: checkpoints/PSG/yolo11m_psg/weights/best.onnx")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Load saved metrics
results_csv = "checkpoints/PSG/yolo11m_psg/results.csv"
metrics_df = pd.read_csv(results_csv)

# Load summary
with open("checkpoints/PSG/yolo11m_psg/training_summary.json", 'r') as f:
    summary = json.load(f)

print("=" * 80)
print("TRAINING ANALYSIS")
print("=" * 80)
print(f"\nModel: {summary['model']}")
print(f"Dataset: {summary['dataset']}")
print(f"Training Date: {summary['timestamp']}")
print(f"\nConfiguration:")
for key, value in summary['config'].items():
    print(f"  {key}: {value}")

print(f"\n📊 Final Performance:")
for key, value in summary['final_metrics'].items():
    print(f"  {key}: {value:.4f}")

print(f"\n🏆 Best Epoch: {summary['best_epoch']}")
print(f"📈 Total Epochs: {summary['total_epochs_trained']}")

# Statistical summary
print("\n" + "=" * 80)
print("METRICS STATISTICS")
print("=" * 80)
print(metrics_df[['metrics/mAP50(B)', 'metrics/mAP50-95(B)', 
                  'metrics/precision(B)', 'metrics/recall(B)']].describe())

# Correlation analysis
print("\n" + "=" * 80)
print("CORRELATION MATRIX")
print("=" * 80)
corr_cols = ['train/box_loss', 'val/box_loss', 'metrics/mAP50(B)', 
             'metrics/precision(B)', 'metrics/recall(B)']
correlation = metrics_df[corr_cols].corr()
print(correlation)

### Advanced Visualizations

Các biểu đồ chi tiết cho paper/thesis

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")

# Create comprehensive figure
fig = plt.figure(figsize=(20, 12))
gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)

# 1. mAP Evolution (Large plot)
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(metrics_df['epoch'], metrics_df['metrics/mAP50(B)'], 
         label='mAP@0.5', linewidth=2.5, color='#2ecc71')
ax1.plot(metrics_df['epoch'], metrics_df['metrics/mAP50-95(B)'], 
         label='mAP@0.5:0.95', linewidth=2.5, color='#3498db')
ax1.axvline(x=summary['best_epoch'], color='red', linestyle='--', 
            label=f'Best Epoch ({summary["best_epoch"]})', alpha=0.7)
ax1.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax1.set_ylabel('mAP', fontsize=12, fontweight='bold')
ax1.set_title('Mean Average Precision Over Time', fontsize=14, fontweight='bold')
ax1.legend(fontsize=11)
ax1.grid(True, alpha=0.3)

# 2. Loss Comparison
ax2 = fig.add_subplot(gs[0, 2])
box_ax = ax2.twinx()
epochs = metrics_df['epoch']
ax2.plot(epochs, metrics_df['train/box_loss'], label='Train Box', color='#e74c3c', linewidth=2)
ax2.plot(epochs, metrics_df['val/box_loss'], label='Val Box', color='#c0392b', linewidth=2)
box_ax.plot(epochs, metrics_df['train/cls_loss'], label='Train Cls', 
            color='#9b59b6', linewidth=2, linestyle='--')
box_ax.plot(epochs, metrics_df['val/cls_loss'], label='Val Cls', 
            color='#8e44ad', linewidth=2, linestyle='--')
ax2.set_xlabel('Epoch', fontsize=10, fontweight='bold')
ax2.set_ylabel('Box Loss', fontsize=10, fontweight='bold', color='#e74c3c')
box_ax.set_ylabel('Cls Loss', fontsize=10, fontweight='bold', color='#9b59b6')
ax2.set_title('Training Losses', fontsize=12, fontweight='bold')
ax2.legend(loc='upper left', fontsize=9)
box_ax.legend(loc='upper right', fontsize=9)
ax2.grid(True, alpha=0.3)

# 3. Precision vs Recall
ax3 = fig.add_subplot(gs[1, 0])
ax3.plot(metrics_df['epoch'], metrics_df['metrics/precision(B)'], 
         label='Precision', linewidth=2.5, color='#e67e22')
ax3.plot(metrics_df['epoch'], metrics_df['metrics/recall(B)'], 
         label='Recall', linewidth=2.5, color='#f39c12')
ax3.set_xlabel('Epoch', fontsize=10, fontweight='bold')
ax3.set_ylabel('Score', fontsize=10, fontweight='bold')
ax3.set_title('Precision & Recall', fontsize=12, fontweight='bold')
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

# 4. F1-Score (computed)
ax4 = fig.add_subplot(gs[1, 1])
precision = metrics_df['metrics/precision(B)']
recall = metrics_df['metrics/recall(B)']
f1_score = 2 * (precision * recall) / (precision + recall + 1e-6)
ax4.plot(metrics_df['epoch'], f1_score, linewidth=2.5, color='#16a085')
ax4.set_xlabel('Epoch', fontsize=10, fontweight='bold')
ax4.set_ylabel('F1-Score', fontsize=10, fontweight='bold')
ax4.set_title('F1-Score Evolution', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3)

# 5. Learning Rate (if available)
ax5 = fig.add_subplot(gs[1, 2])
if 'lr/pg0' in metrics_df.columns:
    ax5.plot(metrics_df['epoch'], metrics_df['lr/pg0'], linewidth=2.5, color='#d35400')
    ax5.set_xlabel('Epoch', fontsize=10, fontweight='bold')
    ax5.set_ylabel('Learning Rate', fontsize=10, fontweight='bold')
    ax5.set_title('Learning Rate Schedule', fontsize=12, fontweight='bold')
    ax5.set_yscale('log')
    ax5.grid(True, alpha=0.3)
else:
    ax5.text(0.5, 0.5, 'LR data not available', 
             ha='center', va='center', fontsize=12)
    ax5.axis('off')

# 6. Overfitting Analysis
ax6 = fig.add_subplot(gs[2, 0])
train_val_gap = metrics_df['train/box_loss'] - metrics_df['val/box_loss']
ax6.plot(metrics_df['epoch'], train_val_gap, linewidth=2.5, color='#c0392b')
ax6.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax6.fill_between(metrics_df['epoch'], train_val_gap, 0, 
                  where=(train_val_gap > 0), alpha=0.3, color='red', label='Overfitting')
ax6.fill_between(metrics_df['epoch'], train_val_gap, 0, 
                  where=(train_val_gap < 0), alpha=0.3, color='green', label='Underfitting')
ax6.set_xlabel('Epoch', fontsize=10, fontweight='bold')
ax6.set_ylabel('Train - Val Loss', fontsize=10, fontweight='bold')
ax6.set_title('Overfitting Monitor', fontsize=12, fontweight='bold')
ax6.legend(fontsize=9)
ax6.grid(True, alpha=0.3)

# 7. Metrics Distribution (Box plot)
ax7 = fig.add_subplot(gs[2, 1])
data_to_plot = [metrics_df['metrics/mAP50(B)'], 
                metrics_df['metrics/mAP50-95(B)'],
                metrics_df['metrics/precision(B)'],
                metrics_df['metrics/recall(B)']]
bp = ax7.boxplot(data_to_plot, labels=['mAP50', 'mAP50-95', 'Precision', 'Recall'],
                  patch_artist=True)
for patch, color in zip(bp['boxes'], ['#2ecc71', '#3498db', '#e67e22', '#f39c12']):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
ax7.set_ylabel('Score', fontsize=10, fontweight='bold')
ax7.set_title('Metrics Distribution', fontsize=12, fontweight='bold')
ax7.grid(True, alpha=0.3, axis='y')

# 8. Summary Table
ax8 = fig.add_subplot(gs[2, 2])
ax8.axis('off')
summary_text = f"""
TRAINING SUMMARY
{'=' * 30}

Configuration:
  • Model: {summary['model']}
  • Epochs: {summary['config']['epochs']}
  • Batch Size: {summary['config']['batch_size']}
  • Image Size: {summary['config']['img_size']}

Final Performance:
  • mAP@0.5: {summary['final_metrics']['mAP50']:.4f}
  • mAP@0.5:0.95: {summary['final_metrics']['mAP50-95']:.4f}
  • Precision: {summary['final_metrics']['precision']:.4f}
  • Recall: {summary['final_metrics']['recall']:.4f}

Training Info:
  • Best Epoch: {summary['best_epoch']}
  • Total Epochs: {summary['total_epochs_trained']}
"""
ax8.text(0.1, 0.5, summary_text, fontsize=10, verticalalignment='center',
         family='monospace', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.suptitle('YOLO Training Analysis - PSG Dataset', 
             fontsize=16, fontweight='bold', y=0.995)

# Save high-resolution figure for paper
plt.savefig('checkpoints/PSG/yolo11m_psg/training_analysis_detailed.png', 
            dpi=300, bbox_inches='tight')
plt.show()

print("\n✅ Detailed analysis saved to: checkpoints/PSG/yolo11m_psg/training_analysis_detailed.png")

### Compare Multiple Training Runs

So sánh nhiều lần training khác nhau

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import json

# Define multiple runs to compare
runs = {
    "YOLO11m": "checkpoints/PSG/yolo11m_psg/results.csv",
    # "YOLO11s": "checkpoints/PSG/yolo11s_psg/results.csv",
    # "YOLO11x": "checkpoints/PSG/yolo11x_psg/results.csv",
    # Add more runs as needed
}

# Load all runs
all_runs = {}
for name, path in runs.items():
    if os.path.exists(path):
        all_runs[name] = pd.read_csv(path)
        print(f"✓ Loaded: {name}")
    else:
        print(f"⚠ Not found: {path}")

if len(all_runs) > 0:
    # Comparison plots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    colors = ['#2ecc71', '#3498db', '#e74c3c', '#f39c12', '#9b59b6']
    
    # mAP50
    for i, (name, df) in enumerate(all_runs.items()):
        axes[0, 0].plot(df['epoch'], df['metrics/mAP50(B)'], 
                       label=name, linewidth=2.5, color=colors[i % len(colors)])
    axes[0, 0].set_xlabel('Epoch', fontweight='bold')
    axes[0, 0].set_ylabel('mAP@0.5', fontweight='bold')
    axes[0, 0].set_title('mAP@0.5 Comparison', fontweight='bold')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # mAP50-95
    for i, (name, df) in enumerate(all_runs.items()):
        axes[0, 1].plot(df['epoch'], df['metrics/mAP50-95(B)'], 
                       label=name, linewidth=2.5, color=colors[i % len(colors)])
    axes[0, 1].set_xlabel('Epoch', fontweight='bold')
    axes[0, 1].set_ylabel('mAP@0.5:0.95', fontweight='bold')
    axes[0, 1].set_title('mAP@0.5:0.95 Comparison', fontweight='bold')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Validation Loss
    for i, (name, df) in enumerate(all_runs.items()):
        axes[1, 0].plot(df['epoch'], df['val/box_loss'], 
                       label=name, linewidth=2.5, color=colors[i % len(colors)])
    axes[1, 0].set_xlabel('Epoch', fontweight='bold')
    axes[1, 0].set_ylabel('Validation Loss', fontweight='bold')
    axes[1, 0].set_title('Validation Loss Comparison', fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Final metrics comparison (bar chart)
    final_metrics = {}
    for name, df in all_runs.items():
        final_metrics[name] = {
            'mAP50': df['metrics/mAP50(B)'].iloc[-1],
            'mAP50-95': df['metrics/mAP50-95(B)'].iloc[-1],
            'Precision': df['metrics/precision(B)'].iloc[-1],
            'Recall': df['metrics/recall(B)'].iloc[-1],
        }
    
    metrics_names = list(list(final_metrics.values())[0].keys())
    x = range(len(metrics_names))
    width = 0.8 / len(all_runs)
    
    for i, (name, metrics) in enumerate(final_metrics.items()):
        values = list(metrics.values())
        axes[1, 1].bar([xi + i*width for xi in x], values, width, 
                      label=name, color=colors[i % len(colors)], alpha=0.8)
    
    axes[1, 1].set_xlabel('Metrics', fontweight='bold')
    axes[1, 1].set_ylabel('Score', fontweight='bold')
    axes[1, 1].set_title('Final Performance Comparison', fontweight='bold')
    axes[1, 1].set_xticks([xi + width*(len(all_runs)-1)/2 for xi in x])
    axes[1, 1].set_xticklabels(metrics_names)
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    plt.suptitle('Multi-Run Training Comparison', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('checkpoints/PSG/training_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print comparison table
    print("\n" + "=" * 80)
    print("FINAL PERFORMANCE COMPARISON")
    print("=" * 80)
    comparison_df = pd.DataFrame(final_metrics).T
    print(comparison_df.to_string())
    
    # Save comparison
    comparison_df.to_csv('checkpoints/PSG/training_comparison.csv')
    print(f"\n✅ Comparison saved to: checkpoints/PSG/training_comparison.csv")
else:
    print("\n⚠ No training runs found to compare")